# Modi's Toxicology (Section 2) – RAG Chatbot
**Source:** *Modi: A Textbook of Medical Jurisprudence and Toxicology*, 25th Ed., Justice K. Kannan, LexisNexis
**Scope:** Section 2 only – Chapters 1 to 12 + Appendix 1 (book pages 3 to 361)

**What this notebook does**
1. Reads your PDF of Section 2 (you supply it, I only had the contents pages)
2. Splits it by the book's own headings: Symptoms, Fatal Dose, Fatal Period, Treatment, Post-mortem Appearance, Medico-legal Points, Biological identification and quantification
3. Tags every chunk with chapter, poison and heading
4. Searches with BM25 + embeddings together (hybrid)
5. Answers with page numbers (works offline; optional Claude for nicer answers)
6. Shows it in a website-style Gradio app with a chapter / poison / heading browser

> Study and forensic reference only. Not clinical advice. In India, for a real poisoning emergency call a hospital or the AIIMS National Poisons Information Centre (1800-116-117).

## 1. Install (run once)

In [1]:
%pip install -q pymupdf sentence-transformers faiss-cpu rank_bm25 gradio numpy anthropic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Config – edit these

In [2]:
import os, re, json
from pathlib import Path

PDF_PATH   = r"C:\Users\praha\Downloads\Adobe Scan 02 Nov 2026 (1).pdf"   # <- your Section 2 PDF
PAGE_OFFSET = 0     # pdf_page_number = book_page + PAGE_OFFSET. Calibrate in step 4.
LOGO_PATH  = "assets/logo.png"           # <- put your own logo file here (optional)
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
USE_CLAUDE = bool(os.getenv("ANTHROPIC_API_KEY"))   # set key in env to enable
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
INDEX_DIR = Path("index_store"); INDEX_DIR.mkdir(exist_ok=True)

## 3. The book's index as structured data
Chapter page ranges come straight from your contents photos. The poison lists are used to tag chunks. Add or fix names any time.

In [3]:
CHAPTERS = {
 1:  dict(title="Poisons and Their Medico-Legal Aspects", pages=(3, 32)),
 2:  dict(title="Corrosive Poisons", pages=(33, 54)),
 3:  dict(title="Inorganic Irritant Poisons (I)", pages=(55, 71)),
 4:  dict(title="Inorganic Irritant Poisons (II) - Metallic Poisons", pages=(73, 145)),
 5:  dict(title="Organic Irritant Poisons (I) - Vegetable Poisons", pages=(147, 171)),
 6:  dict(title="Organic Plant Poisons", pages=(173, 194)),
 7:  dict(title="Organic Irritant Poisons - Animal Poisons", pages=(195, 211)),
 8:  dict(title="Somniferous Cerebral Poisons", pages=(213, 229)),
 9:  dict(title="Inebriant Cerebral Poisons", pages=(231, 284)),
 10: dict(title="Deliriant Cerebral Poisons", pages=(285, 298)),
 11: dict(title="Asphyxiants (Irrespirable Gases)", pages=(299, 313)),
 12: dict(title="Mycotoxins", pages=(315, 321)),
 "A1": dict(title="Appendix 1 - Somniferous Cerebral Poisons: Analytical Methodology", pages=(323, 360)),
}

# Headings the book repeats under each poison
SECTION_LABELS = [
 "Properties","Special Symptoms","Symptoms","Acute Poisoning","Chronic Poisoning","Fatal Dose and Fatal Period",
 "Fatal Dose","Fatal Period","Diagnosis","Biological identification and quantification","Biological quantification",
 "Treatment","Post-mortem Appearance","Medico-legal Points","Medico-legal Aspects","Pathophysiology",
 "Irritant Symptoms","Neurotic Symptoms","Tests",
]

# Poisons / topics per chapter (names + common Indian names) used for tagging
POISONS = {
 1: ["Concept and Scope of Toxicology","Law Relating to Poisons","Narcotic Drugs and Psychotropic Substances Act","Drugs and Cosmetics Act","Pharmacy Act","Classification of Poisons","Routes of Administration","Actions of Poisons","Diagnosis of Poisoning","Chemical Analysis","Examination of the Viscera","Fatal Doses","Treatment in Cases of Poisoning","Use of Antidotes","Poisoning Center","Case Law on Poisoning"],
 2: ["Sulphuric Acid","Nitric Acid","Hydrochloric Acid","Hydrofluoric Acid","Oxalic Acid","Carbolic Acid","Phenol","Creosote","Thymol","Picric Acid","Salicylic Acid","Aspirin","Acetic Acid","Tartaric Acid","Citric Acid","Ammonia","Potassium Hydroxide","Sodium Hydroxide","Ammonium Carbonate","Potassium Carbonate","Sodium Carbonate","Calcium Hydroxide","Vitriol throwing"],
 3: ["Phosphorus","Phosphine","Organophosphorus","Triorthocresyl Phosphate","Chlorine","Bromine","Iodine","Boric Acid","Borax"],
 4: ["Arsenic","Antimony","Mercury","Copper","Lead","Thallium","Zinc","Bismuth","Silver","Iron","Manganese","Tin","Chromium","Potassium Nitrate","Potassium Chlorate","Alum","Magnesium","Barium","Sodium Chloride","Sodium Nitrite","Beryllium","Cadmium","Gold","Platinum","Nickel","Cobalt","Osmium"],
 5: ["Castor","Ricin","Croton","Jequirity","Abrus","Bitter Apple","Colocynth","Ergot","Chillies","Marking-Nut","Semecarpus","Madar","Calotropis","Plumbago","Veratrum","Colchicum","Stavesacre","Hellebore","Laburnum","Yew","Savin","Gamboge","Kaladana","Euphorbium","Cleistanthus","Physic Nut","Jatropha","Aloes","Karihari","Gloriosa","Cuckoo-Pint","Argemone","Kakmari","Cocculus","Sharagava","Satap","Rhitha","Bahera","Oleander","Licorice","Ephedra","Hydrocyanic Acid","Hydrogen Cyanide","Prussic Acid"],
 6: ["Tobacco","Nicotine","Lobelia","Jaborandi","Areca","Digitalis","Strophanthus","Quinine","Nerium","Thevetia","Cerbera","Aconite","Hemlock","Coniine","Curare","Strychnine","Nux Vomica","Kuchila","Brucine","Calabar Bean","Physostigmine","Gelsemium","Jasmine"],
 7: ["Cantharides","Snake","Cobra","Krait","Viper","Antivenom","Insects","Scorpion","Centipede","Food Poisoning","Botulism","Fish Poisoning","Ptomaines","Insulin"],
 8: ["Opium","Morphine","Heroin","Codeine","Fentanyl","Opium Addiction"],
 9: ["Alcohol","Ethyl Alcohol","Widmark","Motor Vehicles Act","Methyl Alcohol","Methanol","Methyl Chloride","Methyl Bromide","Ethylene Glycol","Amyl Alcohol","Amyl Nitrite","Metaldehyde","Formaldehyde","Ether","Ethyl Chloride","Chloroform","Carbon Tetrachloride","Tetrachlorethane","Trichlorethylene","DDT","Endrin","Bromoform","Iodoform","Chloral Hydrate","Paraldehyde","Sulphonal","Barbiturates","Veronal","Luminal","Mandrax","Thalidomide","Tranquillisers","Antihistamines","Amphetamine","Sulphonamide","Aniline","Phenylenediamine","Pyridine","Naphthalene","Benzene","Nitrobenzene","Dinitrobenzene","Dinitrophenol","Dinitrocresol","Trinitrotoluene","Nitroglycerin","Petroleum","Turpentine","Eucalyptus Oil","Nutmeg"],
 10: ["Dhatura","Datura","Belladonna","Atropine","Homatropine","Henbane","Hyoscyamus","Indian Hemp","Cannabis","Bhang","Ganja","Charas","Peyote","Mescal","Cocaine","Worm-Wood","Absinthe","Camphor","Lathyrus","Lolium","Paspalum"],
 11: ["Carbon Dioxide","Carbon Monoxide","Carbon Disulphide","Hydrogen Sulphide","Nitrous Oxide","Sulphur Dioxide","Tear Gases","Tissue Asphyxiants","Hydrocyanic Acid"],
 12: ["Mycotoxins","Mycotoxicoses","Trichothecenes","Aflatoxin","Mushroom Poisoning","Poisonous Mushrooms in India","Amantine","Amanita"],
 "A1": ["Analytical Methodology","Somniferous Cerebral Poisons"],
}

# Fully itemised example: Chapter 12 exactly as in your index (book page numbers)
CH12_INDEX = [
 ("Mycotoxins",315),("Some Recognised Mycotoxicoses before 1960",315),("Some Common Mycotoxins",316),
 ("Trichothecenes",316),("Aflatoxin",317),("Mushroom Poisoning",318),("Poisonous Mushrooms in India",319),
 ("Symptoms",320),("Irritant Symptoms",320),("Neurotic Symptoms",320),("Fatal Dose",320),("Fatal Period",320),
 ("Biological Identification and Quantification of Amantine",320),("Treatment",320),
 ("Post-mortem Appearance",320),("Medico-legal Points",320),
 ("Appendix 1 Somniferous Cerebral Poisons: Analytical Methodology",323),("Subject Index",361),
]

def chapter_of(book_page):
    for k,v in CHAPTERS.items():
        a,b = v["pages"]
        if a <= book_page <= b: return k
    return None
print(len(CHAPTERS), "chapters/appendix loaded")

13 chapters/appendix loaded


## 4. Load the PDF and check the page offset

In [4]:
import fitz  # PyMuPDF
doc = fitz.open(PDF_PATH)
print("PDF pages:", len(doc))

def page_text(book_page):
    i = book_page + PAGE_OFFSET - 1          # 0-based
    if not (0 <= i < len(doc)): return ""
    t = doc[i].get_text("text")
    if len(t.strip()) < 30:                  # scanned page -> automatic OCR fallback
        try: t = ocr_page(book_page)
        except Exception as e: pass
    return t

# Calibration: this should show the start of Chapter 1 (Poisons and Their Medico-Legal Aspects)
print(page_text(3)[:400])


PDF pages: 35



c:\Users\praha\anaconda3\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


If page 3 above is **not** Chapter 1, change `PAGE_OFFSET` in step 2 and re-run. Scanned (image-only) PDFs give empty text. See the OCR cell below.

In [5]:
# OCR for scanned PDFs. One-time setup: %pip install pytesseract pillow  AND install Tesseract for Windows
# (https://github.com/UB-Mannheim/tesseract/wiki). Then uncomment the next line if it is not on PATH:
# import pytesseract; pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
_ocr_cache = {}
def ocr_page(book_page):
    import pytesseract, io
    from PIL import Image
    i = book_page + PAGE_OFFSET - 1
    if not (0 <= i < len(doc)): return ""
    if i not in _ocr_cache:
        pix = doc[i].get_pixmap(dpi=250)
        _ocr_cache[i] = pytesseract.image_to_string(Image.open(io.BytesIO(pix.tobytes("png"))))
    return _ocr_cache[i]


## 5. Chunk by the book headings

In [6]:
LABEL_RE = re.compile(r"^\s*(" + "|".join(re.escape(l) for l in sorted(SECTION_LABELS, key=len, reverse=True)) + r")\s*[:.\-]?\s*$", re.I)

def find_poison(line, chapter):
    l = line.strip().lower()
    if len(l) < 4 or len(l) > 140: return None
    for name in POISONS.get(chapter, []):
        if name.lower() in l: return name
    return None

def build_chunks(use_ocr=False, max_chars=1400, overlap=150):
    chunks = []
    for ch, meta in CHAPTERS.items():
        a, b = meta["pages"]
        poison, section = None, "General"
        buf, buf_pages = [], set()
        def flush():
            nonlocal buf, buf_pages
            text = " ".join(buf).strip()
            if len(text) > 80:
                for s in range(0, len(text), max_chars - overlap):
                    piece = text[s:s + max_chars]
                    chunks.append(dict(chapter=str(ch), chapter_title=meta["title"], poison=poison or "General",
                                       section=section, pages=sorted(buf_pages), text=piece))
            buf, buf_pages = [], set()
        for p in range(a, b + 1):
            t = ocr_page(p) if use_ocr else page_text(p)
            for line in t.splitlines():
                if not line.strip(): continue
                if LABEL_RE.match(line):
                    flush(); section = LABEL_RE.match(line).group(1).title(); continue
                pn = find_poison(line, ch)
                if pn and pn != poison and len(line.split()) <= 14:
                    flush(); poison, section = pn, "General"
                buf.append(line.strip()); buf_pages.add(p)
        flush()
    return chunks

CHUNKS = build_chunks()
if not CHUNKS:
    print(f"WARNING: no body text found (PDF has {len(doc)} pages, Section 2 body needs book pages 3-360).")
    print("Running in INDEX-ONLY mode: chapters, topics and page pointers from the book's contents.")
    print("Add the Section 2 BODY PDF in Step 2 for full answers.\n")
    for ch, meta in CHAPTERS.items():
        a_, b_ = meta["pages"]
        for name in POISONS.get(ch, []):
            CHUNKS.append(dict(chapter=str(ch), chapter_title=meta["title"], poison=name, section="Contents",
                               pages=[a_, b_], text=f"Topic '{name}' is covered in Chapter {ch}: {meta['title']}, book pages {a_}-{b_}."))
    for name, p in CH12_INDEX:
        CHUNKS.append(dict(chapter="12" if p < 322 else "A1", chapter_title="Mycotoxins" if p < 322 else CHAPTERS["A1"]["title"],
                           poison=name, section="Contents", pages=[p],
                           text=f"'{name}' appears in the book index at page {p}."))
print(len(CHUNKS), "chunks")
import collections
print(collections.Counter(c["chapter"] for c in CHUNKS))

Running in INDEX-ONLY mode: chapters, topics and page pointers from the book's contents.
Add the Section 2 BODY PDF in Step 2 for full answers.

270 chunks
Counter({'9': 50, '5': 44, '4': 27, '12': 24, '2': 23, '6': 23, '10': 21, '1': 16, '7': 14, '3': 9, '11': 9, '8': 6, 'A1': 4})


## 6. Hybrid index (BM25 + embeddings)

In [7]:
assert CHUNKS, "CHUNKS is empty - fix Step 4/5 first (wrong PDF, PAGE_OFFSET, or OCR not set up)."
import numpy as np, faiss, pickle
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
def enrich(c): return f"Chapter {c['chapter']} {c['chapter_title']} | {c['poison']} | {c['section']}\n{c['text']}"
texts = [enrich(c) for c in CHUNKS]

emb = embedder.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True).astype("float32")
faiss_index = faiss.IndexFlatIP(emb.shape[1]); faiss_index.add(emb)
tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = BM25Okapi([tok(t) for t in texts])
pickle.dump(CHUNKS, open(INDEX_DIR/"chunks.pkl","wb")); faiss.write_index(faiss_index, str(INDEX_DIR/"faiss.idx"))
print("index ready")

W1103 00:34:13.144000 18540 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

index ready


In [8]:
def search(query, k=6, chapter=None, poison=None, section=None):
    n = len(CHUNKS)
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    sims, ids = faiss_index.search(q, min(n, 60))
    dense = {int(i): r for r, i in enumerate(ids[0])}
    bs = bm25.get_scores(tok(query)); sparse = {int(i): r for r, i in enumerate(np.argsort(-bs)[:60])}
    scores = {}
    for i in set(dense) | set(sparse):                     # reciprocal rank fusion
        scores[i] = 1/(60+dense.get(i, 999)) + 1/(60+sparse.get(i, 999))
    out = []
    for i, s in sorted(scores.items(), key=lambda x: -x[1]):
        c = CHUNKS[i]
        if chapter and c["chapter"] != str(chapter): continue
        if poison and poison.lower() not in c["poison"].lower(): continue
        if section and section.lower() not in c["section"].lower(): continue
        out.append((s, c))
        if len(out) == k: break
    return out

for s, c in search("fatal dose of aflatoxin", k=2):
    print(c["chapter"], c["poison"], c["section"], c["pages"], c["text"][:150], "\n")

12 Fatal Dose Contents [320] 'Fatal Dose' appears in the book index at page 320. 

12 Aflatoxin Contents [315, 321] Topic 'Aflatoxin' is covered in Chapter 12: Mycotoxins, book pages 315-321. 



## 7. Answer generation

In [9]:
SYSTEM = ("You answer ONLY from the supplied excerpts of Modi's Medical Jurisprudence and Toxicology (Section 2). "
          "Be clear, in plain English, structured by: Symptoms / Fatal Dose / Fatal Period / Treatment / Post-mortem Appearance / Medico-legal Points when relevant. "
          "Cite book page numbers like (p. 320). If the excerpts do not contain the answer, say so. This is for study and forensic reference, not clinical advice.")

def format_sources(hits):
    return "\n".join(f"- Ch {c['chapter']} | {c['poison']} | {c['section']} | pp. {c['pages'][0]}-{c['pages'][-1]}" for _, c in hits)

def answer(query, chapter=None, poison=None, section=None, k=6):
    hits = search(query, k=k, chapter=chapter, poison=poison, section=section)
    if not hits: return "No matching passage found. Try removing filters."
    if USE_CLAUDE:
        import anthropic
        ctx = "\n\n".join(f"[Ch {c['chapter']} | {c['poison']} | {c['section']} | pp. {c['pages'][0]}-{c['pages'][-1]}]\n{c['text']}" for _, c in hits)
        r = anthropic.Anthropic().messages.create(model=CLAUDE_MODEL, max_tokens=1200, system=SYSTEM,
              messages=[{"role":"user","content":f"Excerpts:\n{ctx}\n\nQuestion: {query}"}])
        return r.content[0].text + "\n\n**Sources**\n" + format_sources(hits)
    body = "\n\n".join(f"**{c['poison']} - {c['section']}** (pp. {c['pages'][0]}-{c['pages'][-1]})\n{c['text']}" for _, c in hits[:4])
    return body + "\n\n**Sources**\n" + format_sources(hits)

print(answer("What are the post-mortem appearances in mushroom poisoning?", chapter=12)[:800])

**Post-mortem Appearance - Contents** (pp. 320-320)
'Post-mortem Appearance' appears in the book index at page 320.

**Mushroom Poisoning - Contents** (pp. 318-318)
'Mushroom Poisoning' appears in the book index at page 318.

**Mushroom Poisoning - Contents** (pp. 315-321)
Topic 'Mushroom Poisoning' is covered in Chapter 12: Mycotoxins, book pages 315-321.

**Poisonous Mushrooms in India - Contents** (pp. 319-319)
'Poisonous Mushrooms in India' appears in the book index at page 319.

**Sources**
- Ch 12 | Post-mortem Appearance | Contents | pp. 320-320
- Ch 12 | Mushroom Poisoning | Contents | pp. 318-318
- Ch 12 | Mushroom Poisoning | Contents | pp. 315-321
- Ch 12 | Poisonous Mushrooms in India | Contents | pp. 319-319
- Ch 12 | Fatal Period | Contents | pp. 320-320
- Ch 12 | Fatal Dose 


## 8. Website-style app
Browse by **Chapter -> Poison -> Heading** (exactly like the book's index) or just ask a question. Put your own logo at `LOGO_PATH` (use only logos you have the right to use).

In [10]:
import gradio as gr
from collections import defaultdict

tree = defaultdict(lambda: defaultdict(set))
for c in CHUNKS: tree[c["chapter"]][c["poison"]].add(c["section"])
ch_choices = [f"{k} - {v['title']}" for k, v in CHAPTERS.items()]
ch_key = lambda s: None if not s else s.split(" - ")[0]

CSS = """
.hdr{background:#14213d;color:#fff;padding:18px 24px;border-radius:10px;display:flex;align-items:center;gap:16px}
.hdr h1{margin:0;font-size:22px} .hdr p{margin:2px 0 0;opacity:.8;font-size:13px}
.warn{background:#fff4e5;border-left:4px solid #fca311;padding:8px 12px;font-size:12px;border-radius:6px}
"""
def on_chapter(c):
    k = ch_key(c); ps = sorted(tree[k].keys()) if k else []
    return gr.update(choices=ps, value=None), gr.update(choices=[], value=None)
def on_poison(c, p):
    k = ch_key(c); ss = sorted(tree[k][p]) if k and p else []
    return gr.update(choices=ss, value=None)
def respond(msg, history, c, p, s):
    history = history + [(msg, answer(msg, ch_key(c), p, s))]
    return history, ""

with gr.Blocks(css=CSS, title="Modi Toxicology Assistant") as app:
    logo = f"<img src='file/{LOGO_PATH}' height='44'>" if os.path.exists(LOGO_PATH) else "&#9879;"
    gr.HTML(f"<div class='hdr'>{logo}<div><h1>Modi's Toxicology Assistant</h1><p>Section 2 - Chapters 1-12 and Appendix 1 | Medical Jurisprudence and Toxicology, 25th Ed.</p></div></div>")
    gr.HTML("<div class='warn'>Study and forensic reference only. Not medical advice. For a real poisoning emergency contact a hospital or a poison information centre immediately.</div>")
    with gr.Row():
        with gr.Column(scale=1):
            dd_c = gr.Dropdown(ch_choices, label="Chapter")
            dd_p = gr.Dropdown([], label="Poison / topic")
            dd_s = gr.Dropdown([], label="Heading (Symptoms, Fatal Dose ...)")
            gr.Examples([["Fatal dose and fatal period of aflatoxin"],["Treatment of snake bite"],["Medico-legal points of arsenic poisoning"],["Widmark's formula"],["Post-mortem appearance in carbon monoxide poisoning"]], inputs=[gr.Textbox(visible=False)])
        with gr.Column(scale=3):
            chat = gr.Chatbot(height=520, label="Ask the book")
            box = gr.Textbox(placeholder="e.g. Symptoms of Amanita mushroom poisoning", label="Your question")
    dd_c.change(on_chapter, dd_c, [dd_p, dd_s]); dd_p.change(on_poison, [dd_c, dd_p], dd_s)
    box.submit(respond, [box, chat, dd_c, dd_p, dd_s], [chat, box])
app.launch()

C:\Users\praha\AppData\Local\Temp\ipykernel_18540\786361563.py:24: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="Modi Toxicology Assistant") as app:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 9. Next steps
- Check `CHUNKS[i]` for a few poisons; fix names in `POISONS` if a heading is mis-tagged.
- Chunk quality is the biggest lever: if tags look wrong, tighten `find_poison` or add exact page ranges per poison from your index.
- Add a small test set (30 questions with the expected page) and measure hit-rate before tuning.
- Keep the PDF and index private; the book is copyrighted, so do not publish the index or host the app publicly.